In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_IGI Airport (T3), Delhi - IMD.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,Eth-Benzene,RH,WS,WD,BP,Xylene,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,120.48,196.68,36.97,31.09,46.59,0.58,18.55,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,02-01-2025 00:00,03-01-2025 00:00,150.02,247.99,36.70,32.55,47.15,0.73,16.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,03-01-2025 00:00,04-01-2025 00:00,219.69,387.11,110.82,48.68,115.97,1.49,17.54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,04-01-2025 00:00,05-01-2025 00:00,217.37,333.62,56.38,58.29,76.85,1.71,24.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,05-01-2025 00:00,06-01-2025 00:00,135.96,205.47,25.45,29.36,36.32,0.79,19.37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,263.76,500.95,60.90,159.83,85.48,NaN,3.77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
316,13-11-2025 00:00,14-11-2025 00:00,220.52,439.98,32.36,143.57,62.01,NaN,2.31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
317,14-11-2025 00:00,15-11-2025 00:00,173.05,359.47,46.10,151.65,73.38,NaN,2.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
318,15-11-2025 00:00,16-11-2025 00:00,177.81,380.36,67.13,156.70,88.71,NaN,2.49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 11)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Benzene']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 5. Convert date columns to datetime ----------
if 'From Date' in df.columns:
    df['From Date'] = pd.to_datetime(df['From Date'], errors='coerce')
if 'To Date' in df.columns:
    df['To Date'] = pd.to_datetime(df['To Date'], errors='coerce')



# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 10)
   From Date    To Date   PM2.5    PM10     NO    NO2     NOx    CO  Ozone  \
0 2025-01-01 2025-02-01  120.48  196.68  36.97  31.09   46.59  0.97  18.55   
1 2025-02-01 2025-03-01  150.02  247.99  36.70  32.55   47.15  0.97  16.80   
2 2025-03-01 2025-04-01   53.84  149.61  25.44  48.68  115.97  0.97  17.54   
3 2025-04-01 2025-05-01   53.84  333.62  56.38  58.29   76.85  0.97  24.82   
4 2025-05-01 2025-06-01  135.96  205.47  25.45  29.36   36.32  0.97  19.37   

   TOT-RF  
0       0  
1       0  
2       0  
3       0  
4       0  


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,TOT-RF
0,2025-01-01,2025-02-01,2.228596,0.648737,0.339378,-1.722239,-0.107891,0.0,-0.299164,0.0
1,2025-02-01,2025-03-01,3.266797,1.448088,0.326194,-1.619426,-0.083654,0.0,-0.370602,0.0
2,2025-03-01,2025-04-01,-0.113506,-0.084561,-0.223640,-0.483546,2.894839,0.0,-0.340394,0.0
3,2025-04-01,2025-05-01,-0.113506,2.782107,1.287182,0.193194,1.201746,0.0,-0.043213,0.0
4,2025-05-01,2025-06-01,2.772650,0.785675,-0.223152,-1.844066,-0.552371,0.0,-0.265691,0.0
...,...,...,...,...,...,...,...,...,...,...
315,2025-12-11,NaT,-0.113506,-0.084561,1.507896,-0.013842,1.575248,0.0,-0.902508,0.0
316,NaT,NaT,-0.113506,-0.084561,0.114268,-0.013842,0.559479,0.0,-0.962108,0.0
317,NaT,NaT,-0.113506,-0.084561,0.785202,-0.013842,1.051566,0.0,-0.969864,0.0
318,NaT,NaT,-0.113506,-0.084561,1.812111,-0.013842,1.715041,0.0,-0.954760,0.0


In [10]:
df.to_excel('IGIAirport2025.xlsx', index=False)